# 11 — Key-Value Store + Write-Ahead Log (Amazon FAR style)

An in-memory KV store that survives crashes via a write-ahead log. Parts 1-3 build the core
(concurrency at Part 3); Parts 4-5 are stretch tasks (crash recovery, then sharding + parallel
compaction). Fill stubs, run each test cell, peek at the collapsed `ref_` solutions only after trying.

A log entry is `("put", key, value)` or `("delete", key)`.

---

## Part 1 — In-memory store

`KVStore` with `get(k)` (None if absent), `put(k, v)`, `delete(k)` (no error if absent), and
`items()` (a copy of the current mapping).

**Lock down:** last write wins; deleting a missing key is a no-op; `items()` returns a snapshot, not
the live dict.

In [ ]:
class KVStore:
    def __init__(self):
        raise NotImplementedError

    def get(self, k):
        raise NotImplementedError

    def put(self, k, v):
        raise NotImplementedError

    def delete(self, k):
        raise NotImplementedError

    def items(self):
        raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    s = KVStore()
    s.put("a", 1); s.put("b", 2)
    assert s.get("a") == 1
    s.put("a", 3); assert s.get("a") == 3
    s.delete("a"); assert s.get("a") is None
    s.delete("missing")                               # no error
    assert s.items() == {"b": 2}
    print("Part 1 OK")

_t1()

## Part 2 — Replay a log

`replay(ops) -> dict`: rebuild the final state by applying a write-ahead log in order (`("put", k, v)`
sets, `("delete", k)` removes). It's a **pure function of the log** — same log, same state.

**Lock down:** this is how a store rebuilds after restart; ordering matters (last write wins); an empty
log gives an empty store.

In [ ]:
def replay(ops):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    ops = [("put", "a", 1), ("put", "b", 2), ("put", "a", 3), ("delete", "b")]
    assert replay(ops) == {"a": 3}
    assert replay([]) == {}
    assert replay(ops) == replay(ops)                 # deterministic
    print("Part 2 OK")

_t2()

## Part 3 — Thread-safe store

`ConcurrentKVStore`: `get`/`put`/`delete`/`items`, all thread-safe, plus an **atomic** `incr(k)` (set
to `get(k) or 0` plus one). Many threads must be able to write distinct keys and hammer the same
counter without losing updates.

**The invariant:** after 8 threads each `incr("c")` 1000 times, `get("c") == 8000`. **Discuss:** why
`incr` needs the lock around read-modify-write (the classic check-then-act race), coarse vs striped
locking, and that this is coordination work (threads), not CPU work.

In [ ]:
import threading


class ConcurrentKVStore:
    def __init__(self):
        raise NotImplementedError

    def get(self, k):
        raise NotImplementedError

    def put(self, k, v):
        raise NotImplementedError

    def delete(self, k):
        raise NotImplementedError

    def incr(self, k):
        raise NotImplementedError

    def items(self):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    s = ConcurrentKVStore()

    def writer(base):
        for i in range(100):
            s.put("%d-%d" % (base, i), i)

    ws = [threading.Thread(target=writer, args=(b,)) for b in range(8)]
    for t in ws: t.start()
    for t in ws: t.join()
    assert len(s.items()) == 800                      # distinct keys, none lost

    def bump():
        for _ in range(1000):
            s.incr("c")

    bs = [threading.Thread(target=bump) for _ in range(8)]
    for t in bs: t.start()
    for t in bs: t.join()
    assert s.get("c") == 8000                          # no lost increments
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Crash recovery

A crash can leave a torn log: a half-written final entry, or a corrupt record. `recover(log) ->
(state, clean_log)` applies every **well-formed** entry (`("put", k, v)` with arity 3, `("delete", k)`
with arity 2), **skips malformed ones**, and returns the rebuilt state plus the list of entries it
actually applied.

Recovery must be **idempotent**: `recover(clean_log)` reproduces the same state, and replaying the
clean log more than once doesn't change it (put/delete are idempotent under last-write-wins).

**Discuss:** checksums/length-prefix framing to detect torn writes, fsync + log-then-apply ordering,
and periodic checkpoint/snapshot to bound replay time.

In [ ]:
def recover(log):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    log = [("put", "a", 1), ("garbled",), ("put", "b", 2), ("delete", "a"), ("put",)]
    state, clean = recover(log)
    assert state == {"b": 2}                           # malformed entries skipped
    assert recover(clean)[0] == state                  # idempotent recovery
    assert recover(clean + clean)[0] == state          # replaying twice is safe
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Sharded store + parallel compaction

**(a)** `ShardedKVStore(n_shards)` routes keys to `n_shards` independent shards (each its own dict +
lock) to cut contention: `put`/`get`/`delete`, routing by `hash(key) % n_shards`.

**(b)** `parallel_compact(logs, num_procs) -> list[dict]`: compact many shard logs to their final
state in parallel with `ProcessPoolExecutor`; worker is `kvstore_workers.compact_log`.

**Discuss:** per-shard locks vs one global lock, hot-key skew, and compaction as an
embarrassingly-parallel reduce over independent logs.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import kvstore_workers


class ShardedKVStore:
    def __init__(self, n_shards):
        raise NotImplementedError

    def put(self, k, v):
        raise NotImplementedError

    def get(self, k):
        raise NotImplementedError

    def delete(self, k):
        raise NotImplementedError


def parallel_compact(logs, num_procs) -> list:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    s = ShardedKVStore(4)
    s.put("a", 1); s.put("b", 2); s.put("a", 3)
    assert s.get("a") == 3 and s.get("b") == 2 and s.get("zz") is None
    logs = [[("put", "x", 1), ("put", "x", 2)], [("put", "y", 9), ("delete", "y")]]
    assert parallel_compact(logs, 2) == [{"x": 2}, {}]
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Snapshot + truncate the log; bounded recovery time.
- Reader/writer locks (or MVCC / copy-on-write) for read-heavy workloads.
- Group commit, batched fsync; transactions spanning multiple keys.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import threading
from concurrent.futures import ProcessPoolExecutor
import kvstore_workers


class RefKVStore:
    def __init__(self):
        self._d = {}

    def get(self, k):
        return self._d.get(k)

    def put(self, k, v):
        self._d[k] = v

    def delete(self, k):
        self._d.pop(k, None)

    def items(self):
        return dict(self._d)


def ref_replay(ops):
    d = {}
    for op in ops:
        if op[0] == "put":
            d[op[1]] = op[2]
        elif op[0] == "delete":
            d.pop(op[1], None)
    return d


class RefConcurrentKVStore:
    def __init__(self):
        self._d = {}
        self._lock = threading.Lock()

    def get(self, k):
        with self._lock:
            return self._d.get(k)

    def put(self, k, v):
        with self._lock:
            self._d[k] = v

    def delete(self, k):
        with self._lock:
            self._d.pop(k, None)

    def incr(self, k):
        with self._lock:                              # guard the read-modify-write
            self._d[k] = self._d.get(k, 0) + 1

    def items(self):
        with self._lock:
            return dict(self._d)


def ref_recover(log):
    d, clean = {}, []
    for entry in log:
        if not isinstance(entry, tuple) or len(entry) < 2:
            continue
        if entry[0] == "put" and len(entry) == 3:
            d[entry[1]] = entry[2]
            clean.append(entry)
        elif entry[0] == "delete" and len(entry) == 2:
            d.pop(entry[1], None)
            clean.append(entry)
        # anything else is malformed -> skip
    return d, clean


class RefShardedKVStore:
    def __init__(self, n_shards):
        self.n = n_shards
        self.shards = [{} for _ in range(n_shards)]
        self.locks = [threading.Lock() for _ in range(n_shards)]

    def _i(self, k):
        return hash(k) % self.n

    def put(self, k, v):
        i = self._i(k)
        with self.locks[i]:
            self.shards[i][k] = v

    def get(self, k):
        i = self._i(k)
        with self.locks[i]:
            return self.shards[i].get(k)

    def delete(self, k):
        i = self._i(k)
        with self.locks[i]:
            self.shards[i].pop(k, None)


def ref_parallel_compact(logs, num_procs):
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        return list(ex.map(kvstore_workers.compact_log, logs))


_s = RefKVStore(); _s.put("a", 1); _s.delete("a"); assert _s.get("a") is None
assert ref_replay([("put", "a", 1), ("delete", "a"), ("put", "a", 2)]) == {"a": 2}
_st, _cl = ref_recover([("put", "a", 1), ("oops",), ("delete", "a")])
assert _st == {} and ref_recover(_cl)[0] == {}
assert ref_parallel_compact([[("put", "k", 9)]], 2) == [{"k": 9}]
print("reference solutions OK")